In [2]:
from torch.utils.data import Dataset, DataLoader
from torchvision.datasets import ImageFolder
from torchvision import transforms
from torchvision import models
import torch
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score 
from sklearn.model_selection import train_test_split
import numpy as np

from torch.optim import lr_scheduler
#os.environ['KMP_DUPLICATE_LIB_OK']='True'

In [3]:
transform = transforms.Compose([transforms.Resize((256, 170)), transforms.ToTensor()])

In [4]:
ds = ImageFolder("C:/Users/khush/Documents/Khushi/4th_Sem/All_India_Woman_Hackathon/archive/Alzheimers-ADNI/both", transform = transform)

In [5]:
train_ds, test_ds = train_test_split(ds, test_size=0.01, random_state = 32)

In [7]:
device = torch.device('cpu')

In [8]:
model = models.resnet50(pretrained=True)
model.fc = torch.nn.Linear(2048, 3)
model = model.to(device) 

LR = 0.01
BATCH_SIZE = 32

optimizer = torch.optim.Adam(model.parameters(), lr = LR, weight_decay = 0.1)
scheduler = lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

criterion = torch.nn.CrossEntropyLoss() 

train_dataloader = DataLoader(train_ds, batch_size = BATCH_SIZE, shuffle=True)
test_dataloader = DataLoader(test_ds, batch_size = BATCH_SIZE, shuffle=True)

C:\Users\khush\anaconda3\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\khush\anaconda3\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [12]:
for im,label in test_dataloader :
    im=im.to(torch.device('cpu'))
    print(im,label)

TypeError: cannot unpack non-iterable int object

In [13]:
EPOCHS = 2

max_acc = 0
history_loss = []
history_acc = []

for e in range(EPOCHS):
    print('EPOCH', e+1, '\n')
    LOSS = 0
    model.train()
    for images, labels in tqdm(train_dataloader, desc='Training'):
        
        images = images.to(device)
        labels = labels.to(device) 
        
        optimizer.zero_grad()
        outputs = model(images) 
        if True:
            try:loss = criterion(outputs, labels)
            except:
                continue
        else:
            loss = criterion_DRW(outputs, labels)
        LOSS += loss.item()
        loss.backward()
        optimizer.step()
        scheduler.step() 
        
        
    LOSS = LOSS/len(train_dataloader)
    history_loss.append(LOSS)
    model.eval()
    
    targets = []
    predictions = [] 
    
    for images,labels in tqdm(test_dataloader, desc='Validation'):
        
        images = images.to(device)
        
        outputs = model(images)
        outputs = torch.argmax(outputs, dim=1).cpu()
        
        targets.extend(labels) 
        predictions.extend(outputs) 
    
    
    acc = accuracy_score(targets, predictions)
    history_acc.append(acc)
    if acc > max_acc:
        max_acc = acc 
        torch.save(model.state_dict(), 'resnet50_best_ckpt.pth')
    torch.save(model.state_dict(), 'resnet50_ckpt.pth')
    print('Loss: ', LOSS, ' Accuracy: ', round(acc*100,2), 'Best: ', round(max_acc*100,2)) 


EPOCH 1 



Training:   0%|          | 0/36 [00:00<?, ?it/s]

Validation:   0%|          | 0/1 [00:00<?, ?it/s]

Loss:  0.0  Accuracy:  25.0 Best:  25.0
EPOCH 2 



Training:   0%|          | 0/36 [00:00<?, ?it/s]

Validation:   0%|          | 0/1 [00:00<?, ?it/s]

Loss:  0.0  Accuracy:  25.0 Best:  25.0


In [14]:
targets = [] 
predictions = [] 


for im, label in tqdm(test_dataloader):
    
    im = im.to(torch.device('cpu'))
    
    outputs = model(im)
    outputs = torch.argmax(outputs, dim=1).cpu()
    
    predictions.extend(outputs)
    targets.extend(label)
    
print(accuracy_score(targets, predictions))

  0%|          | 0/1 [00:00<?, ?it/s]

0.25


In [15]:
test_dataloader

In [23]:
from PIL import Image
PATH_TO_MODEL='C:/Users/khush/Documents/Khushi/4th_Sem/All_India_Woman_Hackathon/resnet50_best_ckpt.pth'
PATH_TO_IMAGE='C:/Users/khush/Documents/Khushi/4th_Sem/All_India_Woman_Hackathon/archive/Alzheimers-ADNI/test/Final AD JPEG/ADNI_002_S_4229_MR_Axial_T2_STAR__br_raw_20180920102955004_22_S728699_I1050348.jpg'

model1 = models.resnet50(pretrained=True)
model1.fc = torch.nn.Linear(2048, 3)
model1 = model1.to(device) 

model1.load_state_dict(torch.load(PATH_TO_MODEL)) 
model1.eval()  

transform = transforms.Compose([transforms.Resize((256, 170)), transforms.ToTensor()]) 

img = Image.open(PATH_TO_IMAGE)  
x = transform(img)  
x = x.unsqueeze(0)  

output = model1(x)  
pred = torch.argmax(output, 1)  
print('Image predicted as ', pred)

RuntimeError: Given groups=1, weight of size [64, 3, 7, 7], expected input[1, 1, 256, 170] to have 3 channels, but got 1 channels instead

In [24]:
PATH_TO_MODEL='C:/Users/khush/Documents/Khushi/4th_Sem/All_India_Woman_Hackathon/resnet50_best_ckpt.pth'
PATH_TO_IMAGE='C:/Users/khush/Documents/Khushi/4th_Sem/All_India_Woman_Hackathon/archive/Alzheimers-ADNI/test/Final AD JPEG/ADNI_002_S_4229_MR_Axial_T2_STAR__br_raw_20180920102955004_22_S728699_I1050348.jpg'

model1 = models.resnet50(pretrained=True)
model1.fc = torch.nn.Linear(2048, 3)
model1 = model1.to(device) 

model1.load_state_dict(torch.load(PATH_TO_MODEL)) 
model1.eval()  


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [26]:
transform = transforms.Compose([transforms.Resize((256, 170)), transforms.ToTensor()])  # Same as for your validation data, e.g. Resize, ToTensor, Normalize, ...

img = Image.open(PATH_TO_IMAGE)  
x = transform(img)  
x = x.unsqueeze(0) 
x

tensor([[[[0.0157, 0.0196, 0.0196,  ..., 0.0196, 0.0157, 0.0157],
          [0.0157, 0.0196, 0.0196,  ..., 0.0196, 0.0157, 0.0157],
          [0.0157, 0.0196, 0.0196,  ..., 0.0196, 0.0196, 0.0157],
          ...,
          [0.0157, 0.0157, 0.0157,  ..., 0.0196, 0.0196, 0.0157],
          [0.0157, 0.0157, 0.0157,  ..., 0.0196, 0.0196, 0.0157],
          [0.0157, 0.0157, 0.0157,  ..., 0.0157, 0.0157, 0.0157]]]])

In [29]:
x.shape

torch.Size([1, 1, 256, 170])

In [30]:
x.permute(0,1,2,3).shape

torch.Size([1, 1, 256, 170])

In [31]:
x.permute(0,1,2,3).shape

RuntimeError: shape '[1, 3, 256, 170]' is invalid for input of size 43520

ModuleNotFoundError: No module named 'tf'

In [33]:
img = Image.open(PATH_TO_IMAGE).convert('RGB')  # Load image as PIL.Image
x = transform(img)  # Preprocess image
x = x.unsqueeze(0)  # Add batch dimension
x.shape

torch.Size([1, 3, 256, 170])

In [34]:
x.permute(0,1,2,3).shape

torch.Size([1, 3, 256, 170])

In [35]:
x.permute(0,3,1,2).shape

torch.Size([1, 170, 3, 256])

In [36]:
output = model1(x)
pred = torch.argmax(output, 1)
print('Image predicted as ', pred)

Image predicted as  tensor([1])


In [38]:
from PIL import Image
PATH_TO_MODEL='C:/Users/khush/Documents/Khushi/4th_Sem/All_India_Woman_Hackathon/resnet50_best_ckpt.pth'
PATH_TO_IMAGE='C:/Users/khush/Documents/Khushi/4th_Sem/All_India_Woman_Hackathon/archive/Alzheimers-ADNI/test/Final EMCI JPEG/ADNI_041_S_4271_MR_Axial_T2_STAR__br_raw_20171002154051755_22_S616695_I913404.jpg'

model1 = models.resnet50(pretrained=True)
model1.fc = torch.nn.Linear(2048, 3)
model1 = model1.to(device) 

model1.load_state_dict(torch.load(PATH_TO_MODEL))  
model1.eval() 

transform = transforms.Compose([transforms.Resize((256, 170)), transforms.ToTensor()]) 

img = Image.open(PATH_TO_IMAGE).convert('RGB') 
x = transform(img)  
x = x.unsqueeze(0)  

output = model1(x)  
pred = torch.argmax(output, 1)  
print('Image predicted as ', pred)

Image predicted as  tensor([1])
